## Benchmark of latents

Load libraries

In [ ]:
# Path-related libraries
import os
import sys
from pyhere import here  # For reproducible relative paths

# Model-related libraries
from sklearn.preprocessing import LabelEncoder  # For encoding categorical variables

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical and tensor operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd
import optuna # An automatic hyperparameter optimization software framework,
import torch              # Core PyTorch library for tensor operations
import torch.nn as nn     # Neural network components
from torch.optim import Adam  # Optimizer for training models
import torch.nn.functional as F
from torch.distributions import Distribution, Gamma
from torch.distributions import Poisson as PoissonTorch
import warnings
from typing import Optional, Union
from torch.distributions.utils import (
    broadcast_all,
    lazy_property,
    logits_to_probs,
    probs_to_logits,
)
import copy               # For deep copying model states

# Benchmark
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [ ]:
# Saving
base_dir = str(here('data/integrate/third_pass/'))
files_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
tmp_dir = os.path.join(base_dir, 'tmp') 
model_dir = os.path.join(base_dir, 'models') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata')) 

GPU enbabled neighborhood search

In [ ]:
from cuvs.neighbors import cagra
from scib_metrics.nearest_neighbors import NeighborsResults
import cupy as cp

def cagra_nn(X: np.ndarray, k: int):
    # Setup data
    X = cp.asarray(np.ascontiguousarray(X, dtype=np.float32))

    # Build index
    index_params = cagra.IndexParams(graph_degree = 128)
    index = cagra.build(index_params, X)

    # Search
    search_params = cagra.SearchParams(itopk_size = 128)
    distances, neighbors = cagra.search(search_params, index, X, k)
    
    # Clean and return
    del index
    return NeighborsResults(indices=cp.asnumpy(neighbors), distances=np.sqrt(cp.asnumpy(distances)))

In [ ]:
# hvgs 
hvgs = [1000, 2000, 4000]

technical_seperate = ['cell_nuclei', 'ic_id_study', 'library_prep', 'ic_id_donor_overall']

celltype = 'study_cell_annotation_harmonized' 

Load adata object

In [ ]:
adata = ad.read_h5ad(anndata_dir, "AF_combined.h5ad"))

Subset to annotated cells

In [ ]:
celltype = "study_cell_annotation_harmonized"

n_before = adata.n_obs
adata_sub = adata[~adata.obs[celltype].isin(["unknown", "excluded"])].copy()
adata_sub.obs[celltype] = adata_sub.obs[celltype].astype("category")
adata_sub.obs[celltype] = adata_sub.obs[celltype].cat.remove_unused_categories()

n_after = adata_sub.n_obs
print(f"Filtered {n_before - n_after} cells (kept {n_after})")

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=FutureWarning,
        message="Argument `use_highly_variable` is deprecated.*"
    )
    
    results = {}
    results_norm = {}
    
    for technical in technical_seperate:

        print(f'Running {technical}')
        
        bm = Benchmarker(
            adata_sub,
            batch_key=technical,
            label_key=celltype,
            bio_conservation_metrics=BioConservation(),
            batch_correction_metrics=BatchCorrection(kbet_per_label = False),
            embedding_obsm_keys=embedding_keys,
            n_jobs=-1,
        )

        bm.prepare(neighbor_computer=cagra_nn)
        bm.benchmark()

        # save results
        results[technical] = bm.get_results(min_max_scale=False)
        results_norm[technical] = bm.get_results(min_max_scale=True)

        bm.get_results(min_max_scale=False).to_csv(os.path.join(file_dir, f'benchmark_celltype_sub_results_{technical}.csv'), index=True)
        bm.get_results(min_max_scale=True).to_csv(os.path.join(file_dir, f'benchmark_celltype_sub_norm_results_{technical}.csv'), index=True)
    
    
        print(f"[OK] Finished benchmark for {technical}")

Load results table and combine into one dataframe

In [ ]:
# Get path to results (not normalised)
result_path = glob.glob(os.path.join(file_dir, "*.csv"))
result_path = [x for x in result_path if "norm" not in x]

df_list = []
for res in result_path:
    # Get base name
    base_name = Path(res).stem.replace("benchmark_celltype_sub_results_", "")
    # Load file
    df_tmp = pd.read_csv(res, index_col=0)
    # Transpose, add variable name and pivot longer
    df_tmp = df_tmp.transpose()
    # Make numeric
    cols = [ 'X_latent_2', 'pca_1000', 'scpoli_1000', 'scvi_1000', 'sysvi_1000']
    df_tmp[cols] = df_tmp[cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop cluster dependet metrics and precalculated aggregate scores
    # As we are not going to cluster based on the KNN, and as these labels are from studies that have not been corrected
    df_tmp = df_tmp.drop(['KMeans NMI', 'KMeans ARI', 'Total', 'Batch correction', 'Bio conservation'])

    # Calculate aggregate scores
    df_means = df_tmp.groupby('Metric Type').mean(numeric_only=True)
    df_means = df_means.transpose()
    df_means['Total'] = 0.5 * df_means["Batch correction"] + 0.5 * df_means["Bio conservation"]
    df_means = df_means.transpose()
    
    # Prepare for combining with df_tmp
    df_means.index.name = 'Embedding'  # to align with df_tmp index
    df_means['Metric Type'] = 'Aggregate score'

    # combine with tmp
    df_tmp = pd.concat([df_tmp, df_means])
    df_tmp["Variable"] = base_name
    df_tmp = df_tmp.rename_axis("Metric").reset_index()
    df_tmp = df_tmp.melt(
        id_vars=["Metric", "Metric Type", "Variable"],
        var_name="Latent",
        value_name="Score",
    )

    # Add to list
    df_list.append(df_tmp)

# Combine results to one df
df_res = pd.concat(df_list)

# round to 4 digits
df_res = df_res.round(2)
print(df_res)

# Save results
df_res.to_csv(os.path.join(file_dir, 'benchmark_results_per_variable.csv'))

Calculat mean aggregate across all variables

In [ ]:
# Mean of aggregated scores
df_total = df_res[
    (df_res["Metric"].isin(["Bio conservation", "Batch correction", "Total"]))
].groupby(['Latent','Metric']).mean(numeric_only=True).round(2)

print(df_total)
df_res.to_csv(os.path.join(file_dir,'benchmark_results_total.csv'))